In [1]:
import os
import sys
import pandas as pd

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from baseline.turbulence_benchmark.utility.turbulence_log_functions import TurbulenceLogHelper

In [4]:
def generate_results_table(res_dir: str):

    """
    This method assumes that all logs are filled. 
    """

    csv_logs = [f for f in os.listdir(res_dir) if 
        os.path.isfile(os.path.join(res_dir, f)) and 
        f.endswith(".csv") and 
        "turbulence" in f.lower()
        ]
    
    print(csv_logs)
    
    if len(csv_logs) != 3:
        raise IndexError("More than 3 Turbulence Benchmark logs in results directory.")
    
    no_mutation_log_name = [f for f in csv_logs if "no_mutation" in f][-1]
    random_log_name = [f for f in csv_logs if "random" in f][-1]
    sequential_log_name = [f for f in csv_logs if "sequential" in f][-1]        
    
    no_mutation_log = pd.read_csv(os.path.join(res_dir, no_mutation_log_name))
    random_log = pd.read_csv(os.path.join(res_dir, random_log_name))
    sequential_log = pd.read_csv(os.path.join(res_dir, sequential_log_name))

    helper = TurbulenceLogHelper()

    res_df = pd.DataFrame(
        index = ["Turbulence", "MuCoCo Random", "MuCoCo Sequential", "MuCoCo Aggregate"],
        columns = ["No. of questions", "No. of tasks"]
    )

    res_df.loc[:, "No. of questions"] = helper.total_questions
    res_df.loc[:, "No. of tasks"] = helper.total_tasks

    turbulence_dict = helper.obtain_turbulence_code_inconsistency_score(no_mutation_log)
    # random_inconsistency_score, random_inconsistency_percentage = helper.obtain_mucoco_code_inconsistency_score(log1= no_mutation_log, log2=random_log)
    # seq_inconsistency_score, seq_inconsistency_percentage = helper.obtain_mucoco_code_inconsistency_score(log1= no_mutation_log, log2=sequential_log)
    random_dict = helper.obtain_turbulence_code_inconsistency_score(log=random_log)
    sequential_dict = helper.obtain_turbulence_code_inconsistency_score(log=sequential_log)

    turbulence_qn_inconsistency_dict = helper.obtain_question_inconsistency_count(log = no_mutation_log)
    random_qn_inconsistency_dict = helper.obtain_question_inconsistency_count(log = random_log)
    sequential_qn_inconsistency_dict = helper.obtain_question_inconsistency_count(log = sequential_log)

    # Updating inconsistency scores
    res_df.loc["Turbulence", "Code Inconsistency Score"] = f"{turbulence_dict['inconsistency_count']}/{turbulence_dict['total_comparisons']}"
    res_df.loc["MuCoCo Random", "Code Inconsistency Score"] = f"{random_dict['inconsistency_count']}/{random_dict['total_comparisons']}"
    res_df.loc["MuCoCo Sequential", "Code Inconsistency Score"] = f"{sequential_dict['inconsistency_count']}/{sequential_dict['total_comparisons']}"
    aggregate_inconsistency_score = (random_dict['inconsistency_count'] + sequential_dict['inconsistency_count'])//2
    res_df.loc["MuCoCo Aggregate", "Code Inconsistency Score"] = f"{aggregate_inconsistency_score}/{sequential_dict['total_comparisons']}"


    # Updating inconsistency percentages
    res_df.loc["Turbulence", "Code Inconsistency %"] = f"{round(turbulence_dict['inconsistency_count']*100 / turbulence_dict['total_comparisons'], 2)}"
    res_df.loc["MuCoCo Random", "Code Inconsistency %"] = f"{round(random_dict['inconsistency_count']*100 / random_dict['total_comparisons'],2 )}"
    res_df.loc["MuCoCo Sequential", "Code Inconsistency %"] = f"{round(sequential_dict['inconsistency_count']*100 / sequential_dict['total_comparisons'], 2)}"
    res_df.loc["MuCoCo Aggregate", "Code Inconsistency %"] = f"{round(aggregate_inconsistency_score*100/sequential_dict['total_comparisons'], 2)}"


    # Updating question inconsistency
    res_df.loc["Turbulence", "Question Inconsistency"] = f"{turbulence_qn_inconsistency_dict['inconsistent_qn_count']}/{turbulence_qn_inconsistency_dict['total_questions']}"
    res_df.loc["MuCoCo Random", "Question Inconsistency"] = f"{random_qn_inconsistency_dict['inconsistent_qn_count']}/{random_qn_inconsistency_dict['total_questions']}"
    res_df.loc["MuCoCo Sequential", "Question Inconsistency"] = f"{sequential_qn_inconsistency_dict['inconsistent_qn_count']}/{sequential_qn_inconsistency_dict['total_questions']}"
    aggregate_qn_inconsistency = (random_qn_inconsistency_dict['inconsistent_qn_count'] + sequential_qn_inconsistency_dict['inconsistent_qn_count'])//2
    res_df.loc["MuCoCo Aggregate", "Question Inconsistency"] = f"{aggregate_qn_inconsistency}/{sequential_qn_inconsistency_dict['total_questions']}"

    # Updating question inconsistency
    res_df.loc["Turbulence", "Question Inconsistency %"] = f"{round(turbulence_qn_inconsistency_dict['inconsistent_qn_count']*100 /turbulence_qn_inconsistency_dict['total_questions'], 2)}"
    res_df.loc["MuCoCo Random", "Question Inconsistency %"] = f"{round(random_qn_inconsistency_dict['inconsistent_qn_count']*100 /random_qn_inconsistency_dict['total_questions'], 2)}"
    res_df.loc["MuCoCo Sequential", "Question Inconsistency %"] = f"{round(sequential_qn_inconsistency_dict['inconsistent_qn_count']*100 /sequential_qn_inconsistency_dict['total_questions'], 2)}"
    res_df.loc["MuCoCo Aggregate", "Question Inconsistency %"] = f"{round(aggregate_qn_inconsistency*100/sequential_qn_inconsistency_dict['total_questions'], 2)}"




    return res_df

res_dir = proj_dir + "/results/code_generation/gpt-4o"

res_dir = "/Users/jin/Downloads/gpt-4o"
generate_results_table(res_dir=res_dir)

['Turbulence_zero_shot_random2.csv', 'Turbulence_zero_shot_sequential2.csv', 'Turbulence_zero_shot_no_mutation2.csv']


,No. of questions,No. of tasks,Code Inconsistency Score,Code Inconsistency %,Question Inconsistency,Question Inconsistency %
Turbulence,52,5204,11051/257806,4.29,18/52,34.62
MuCoCo Random,52,5204,7493/257806,2.91,18/52,34.62
MuCoCo Sequential,52,5204,6955/257806,2.7,14/52,26.92
MuCoCo Aggregate,52,5204,7224/257806,2.8,16/52,30.77
